# Analisi delle occorrenze di una parola target nei romanzi

Scorre tutti i file `.book` in una cartella. Ogni file è un romanzo; ogni riga è un'entità narrativa con:
- `count.occurrence` — quante volte l'entità compare nel testo
- `mentions.proper[0].n` — il nome principale dell'entità
- `agent`, `patient`, `mod`, `poss` — liste di parole associate all'entità

**Obiettivo:** trovare i personaggi nei cui dizionari semantici la parola target compare più spesso, e restituire sia i personaggi sia il romanzo a cui appartengono.

## 0. Configurazione

In [19]:
import ast
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

# ─── PARAMETRI DA MODIFICARE ───────────────────────────────────────────────
CARTELLA      = "/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Memoire/Github_tesi/Corpus_propp"    # percorso della cartella con i file .book
ESTENSIONE    = ".book"
PAROLA_TARGET = "jouer"      # parola da cercare (case-insensitive)
TOP_N         = 10             # quanti risultati restituire

# Campi semantici in cui cercare la parola target
CAMPI_SEMANTICI = ["agent", "patient", "mod", "poss"]
# ───────────────────────────────────────────────────────────────────────────

## 1. Funzioni di parsing

I file `.book` possono usare due formati:
- **JSON** (double quotes) — file più recenti
- **Python dict** (single quotes) — file più vecchi

La funzione `parse_line` gestisce entrambi automaticamente.

In [20]:
def parse_line(line: str) -> dict | None:
    """Parsa una riga provando prima JSON, poi ast.literal_eval."""
    line = line.strip()
    if not line:
        return None
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        pass
    try:
        return ast.literal_eval(line)
    except Exception:
        return None


def conta_parola_in_campi(record: dict, parola: str, campi: list) -> int:
    """
    Conta quante volte 'parola' compare nei campi semantici del record.
    Ogni occorrenza della parola nella lista conta 1.
    """
    parola = parola.lower()
    conteggio = 0
    for campo in campi:
        for item in record.get(campo, []):
            if item.get("w", "").lower() == parola:
                conteggio += 1
    return conteggio


def nome_principale(record: dict) -> str:
    """Restituisce il primo elemento di mentions['proper']['n']."""
    try:
        proper = record["mentions"]["proper"]
        if proper:
            return proper[0]["n"]
    except (KeyError, IndexError, TypeError):
        pass
    return f"entità_{record.get('id', '?')}"


print("Funzioni caricate.")

Funzioni caricate.


## 2. Scansione e conteggio

Per ogni entità si conta **quante volte la parola target appare nei suoi campi semantici** (`agent`, `patient`, `mod`, `poss`). Questo riflette direttamente quanto la parola è associata al personaggio, indipendentemente dalla sua dimensione narrativa generale.

In [21]:
cartella = Path(CARTELLA)
if not cartella.exists():
    raise FileNotFoundError(f"Cartella non trovata: {cartella.resolve()}")

files = sorted(cartella.glob(f"*{ESTENSIONE}"))
print(f"File trovati: {len(files)}\n")

rows = []  # ogni riga = un personaggio con la parola nel suo dizionario

for fp in files:
    titolo = fp.stem
    n_entita_con_parola = 0

    with fp.open(encoding="utf-8", errors="replace") as fh:
        for line in fh:
            record = parse_line(line)
            if record is None:
                continue

            n_occ_parola = conta_parola_in_campi(record, PAROLA_TARGET, CAMPI_SEMANTICI)
            if n_occ_parola == 0:
                continue

            n_entita_con_parola += 1
            rows.append({
                "romanzo":          titolo,
                "personaggio":      nome_principale(record),
                "occ_parola":       n_occ_parola,          # volte che la parola compare nei suoi campi
                "occ_entita":       record.get("count", {}).get("occurrence", 0),  # importanza narrativa
            })

    print(f"  {titolo[:65]:<65}  {n_entita_con_parola:>3} entità con '{PAROLA_TARGET}'")

df = pd.DataFrame(rows)
print(f"\nTotale entità trovate: {len(df)}")

File trovati: 123

  1833_Girardin-Delphine-de_Contes-d-une-vieille-fille-a-ses-neveux    4 entità con 'jouer'
  1835_Woillez-Catherine_Le-Robinson-des-demoiselles                   1 entità con 'jouer'
  1843_Desnoyers-Louis_Les-aventures-de-Jean-Paul-Choppart             4 entità con 'jouer'
  1843_Woillez-Catherine_Leontine-et-Marie-ou-les-Deux-educations      1 entità con 'jouer'
  1845_Dumas-Alexandre_Histoire-d-un-Casse-noisette                    2 entità con 'jouer'
  1846_Musset-Paul-de_Monsieur-le-Vent-et-Madame-la-Pluie              2 entità con 'jouer'
  1848_Woillez-Catherine_Edma-et-Marguerite-ou-les-Ruines-de-Chatil    1 entità con 'jouer'
  1851_Sand-George_Histoire-du-veritable-Gribouille                    1 entità con 'jouer'
  1852_Carraud-Zulma-Tourangin-Mme_La-petite-Jeanne                    1 entità con 'jouer'
  1854_Bassanville-Anais-de_Les-Primeurs-de-la-vie-ou-Bonheurs-joie    3 entità con 'jouer'
  1854_Dumas-Alexandre_La-jeunesse-de-Pierrot                

## 3. Top-N personaggi (con romanzo di appartenenza)

In [22]:
if df.empty:
    print(f"Nessuna entità contiene '{PAROLA_TARGET}' nei campi {CAMPI_SEMANTICI}.")
else:
    top_personaggi = (
        df.sort_values("occ_parola", ascending=False)
        .head(TOP_N)
        [["personaggio", "romanzo", "occ_parola", "occ_entita"]]
        .reset_index(drop=True)
    )
    top_personaggi.index += 1
    top_personaggi.columns = ["personaggio", "romanzo", f"'{PAROLA_TARGET}' nei campi", "occorrenze nel testo"]

    print(f"── Top {TOP_N} personaggi per occorrenze di '{PAROLA_TARGET}' ──")
    display(top_personaggi)

── Top 10 personaggi per occorrenze di 'jouer' ──


,personaggio,romanzo,'jouer' nei campi,occorrenze nel testo
1,mattia,1887_Malot-Hector_Sans-famille,18,6723
2,pascarel,1878_Girardin-Jules_Ouida-Pascarel-roman-imite...,10,2872
3,vitalis,1887_Malot-Hector_Sans-famille,10,7759
4,entità_4,1887_Malot-Hector_Sans-famille,8,772
5,folla,1889_Dombre-Roger_Folla,7,1822
6,capi et moi,1887_Malot-Hector_Sans-famille,6,307
7,sophie,1858_Segur-comtesse-de_Les-Malheurs-de-Sophie,6,3331
8,m. oldham,1894_Gautier-Judith_Memoires-d-un-Elephant-blanc,6,293
9,nane,1905_Toulet-Paul-Jean_Mon-amie-Nane,5,2426
10,raffaëlino,1878_Girardin-Jules_Ouida-Pascarel-roman-imite...,5,1313


## 4. Top-N romanzi

In [14]:
if not df.empty:
    top_romanzi = (
        df.groupby("romanzo", as_index=False)["occ_parola"]
        .sum()
        .sort_values("occ_parola", ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    top_romanzi.index += 1
    top_romanzi.columns = ["romanzo", f"'{PAROLA_TARGET}' totali nei campi"]

    print(f"── Top {TOP_N} romanzi per occorrenze di '{PAROLA_TARGET}' ──")
    display(top_romanzi)

── Top 10 romanzi per occorrenze di 'jouer' ──


,romanzo,'jouer' totali nei campi
1,1887_Malot-Hector_Sans-famille,60
2,1878_Girardin-Jules_Ouida-Pascarel-roman-imite...,32
3,1880_Fleuriot-Zenaide_Tranquille-et-Tourbillon,17
4,1867_Pitray-Olga-de-Segur_Les-enfants-des-Tuil...,17
5,1892_Colomb-Josephine_Les-Conquetes-d-Hermine,16
6,1889_Dombre-Roger_Folla,14
7,1893_Vadier-Berthe_Rose-et-Rosette-odyssee-d-u...,14
8,1894_Gautier-Judith_Memoires-d-un-Elephant-blanc,13
9,1881_Colomb-Josephine_Feu-de-paille,13
10,1866_Muller-Rene_Les-Enfants-gates,13
